# Server Machine Dataset (SMD) Ultra-VETime Teacher Cache Generation
### 38-Channel Multi-Subsystem Teacher Caching & Distillation Across All 28 Servers
- **Domain**: 28 Large-Scale Compute Cluster Servers (708,420 timesteps, 38 metrics, 327 failure events)
- **4 Subsystem Visual Dashboard**: CPU Dynamics (0..7), Memory & Paging (8..14), Network Traffic (15..26), Disk I/O (27..37)
- **Teacher Backbone**: `facebook/dinov2-large` (1024-dim self-supervised representations)
- **Multi-Regime Operational Centroids**: $K=5$ nominal operating clusters
- **Zero Disk I/O Rendering**: Direct Agg memory buffer extraction (<10 ms/window)
- **Multi-GPU Execution**: Parallel batching across provisioned GPUs

In [ ]:
# --- Step 1: Workspace Setup & Codebase Unpacking ---
import os, sys, shutil, subprocess, base64, io, zipfile

repo_dir = '/kaggle/working/VisTAD'
os.makedirs(repo_dir, exist_ok=True)
os.chdir(repo_dir)

print('Unpacking self-contained VisTAD codebase (src, vlm4ts, configs)...')
b64_bundle = "..."
raw_zip = base64.b64decode(b64_bundle)
with zipfile.ZipFile(io.BytesIO(raw_zip), 'r') as z:
    z.extractall(repo_dir)
print('VisTAD codebase unpacked successfully.')
print('Extracted top-level dirs:', os.listdir(repo_dir))


In [ ]:
# --- Step 2: GPU Accelerator Diagnostics ---
import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
num_gpus = torch.cuda.device_count()
print(f'Detected {num_gpus} GPU(s):')
for i in range(num_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {props.name} ({props.total_memory / (1024**3):.1f} GB VRAM)')


In [ ]:
# --- Step 3: SMD Dataset Auto-Discovery & Path Linking ---
import os, glob, shutil
os.makedirs('data/smd/test', exist_ok=True)
os.makedirs('data/smd/test_label', exist_ok=True)

smd_test_candidates = glob.glob('/kaggle/input/**/smd/test/*.txt', recursive=True)
print(f'Found {len(smd_test_candidates)} SMD test files in /kaggle/input')
for p in smd_test_candidates:
    fname = os.path.basename(p)
    target = os.path.join('data/smd/test', fname)
    if not os.path.exists(target):
        try:
            os.symlink(p, target)
        except OSError:
            shutil.copy2(p, target)

smd_label_candidates = glob.glob('/kaggle/input/**/smd/test_label/*.txt', recursive=True)
print(f'Found {len(smd_label_candidates)} SMD test label files in /kaggle/input')
for p in smd_label_candidates:
    fname = os.path.basename(p)
    target = os.path.join('data/smd/test_label', fname)
    if not os.path.exists(target):
        try:
            os.symlink(p, target)
        except OSError:
            shutil.copy2(p, target)

print('Local data/smd/test count:', len(os.listdir('data/smd/test')))
print('Local data/smd/test_label count:', len(os.listdir('data/smd/test_label')))


In [ ]:
# --- Step 4: Execute Ultra-VETime Teacher Cache Generation ---
import os, subprocess, time
os.environ['PYTHONPATH'] = '.'

cmd = [
    'python', 'vlm4ts/generate_smd_cache.py',
    '--data_dir', 'data/smd',
    '--output_train_cache', 'cache/smd_train_ultravetime_cache.pt',
    '--output_eval_cache', 'cache/smd_eval_head_to_head_cache.pt',
    '--window_size', '200',
    '--num_train_windows', '3000',
    '--num_eval_windows', '1500',
    '--model_id', 'facebook/dinov2-large',
    '--n_regimes', '5'
]
print('Executing SMD cache generation command:', ' '.join(cmd))
t0 = time.time()
proc = subprocess.run(cmd, check=True)
dur = time.time() - t0
print(f'SMD Teacher Caching completed successfully in {dur:.1f}s ({dur/60:.2f} min).')


In [ ]:
# --- Step 5: Verify Teacher Cache Statistics ---
import torch, os
import numpy as np

for c_path, name in [
    ('cache/smd_train_ultravetime_cache.pt', 'SMD TRAINING CURRICULUM CACHE'),
    ('cache/smd_eval_head_to_head_cache.pt', 'SMD HEAD-TO-HEAD EVALUATION CACHE')
]:
    if not os.path.exists(c_path):
        print(f'WARNING: {c_path} not found!')
        continue
    sz_mb = os.path.getsize(c_path) / (1024 * 1024)
    c = torch.load(c_path, weights_only=False)
    scores = [v['teacher_score'] for v in c.values()]
    flags = [v['y_flag'] for v in c.values()]
    anoms = sum(flags)
    print(f'\n================== {name} ==================')
    print(f'File Size: {sz_mb:.2f} MB | Total Records: {len(c)}')
    print(f'Anomaly Windows: {anoms} ({anoms/len(c)*100:.1f}%) | Nominal: {len(c)-anoms}')
    print(f'Teacher Anomaly Score: min={min(scores):.4f}, median={np.median(scores):.4f}, max={max(scores):.4f}')
    sample = next(iter(c.values()))
    print(f'Tensor x: {tuple(sample["x"].shape)}')
    print(f'Feature Target z_teacher: {tuple(sample["z_teacher"].shape)}')
    print(f'Dense Mask teacher_dense_mask: {tuple(sample["teacher_dense_mask"].shape)}')


In [ ]:
# --- Step 6: Package Cache Files for Local Retrieval ---
import os, zipfile, glob

out_zip = '/kaggle/working/smd_ultravetime_cache_bundle.zip'
print(f'Packaging cache files into {out_zip}...')
with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob('cache/smd_*.pt'):
        z.write(f, os.path.basename(f))
        print(f'  Added {f} ({os.path.getsize(f)/(1024*1024):.2f} MB)')

print(f'Cache package ready: {out_zip} ({os.path.getsize(out_zip)/(1024*1024):.2f} MB)')
# Also copy files to root working directory for direct download
for f in glob.glob('cache/smd_*.pt'):
    shutil.copy2(f, f'/kaggle/working/{os.path.basename(f)}')
